In [1]:
## 1. Configurar ambiente e caminhos

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Caminhos
BASE_DIR = Path("/Users/chris/Documents/analisis")
XLSX_DIR = BASE_DIR / "exports/hojas/prestacao"  # Prestações de contas XLSX (2022-2026)
CSV_DIR = BASE_DIR / "exports/csv"

# Criar diretórios se não existirem
CSV_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Consolidação de Prestações de Contas (XLSX)")
print(f"  Base: {BASE_DIR}")
print(f"  XLSX (fonte): {XLSX_DIR.exists()}")
print(f"  CSV (saída): {CSV_DIR.exists()}")

# Configurações de processamento
CONFIG = {
    'date_format': '%d/%m/%Y',
    'decimal_sep': ',',
    'thousands_sep': '.',
    'encoding': 'utf-8-sig'
}

✅ Consolidação de Prestações de Contas (XLSX)
  Base: /Users/chris/Documents/analisis
  XLSX (fonte): True
  CSV (saída): True


In [2]:
## 2. Localizar arquivos `.xlsx` (Prestações de Contas)

import re
from datetime import datetime

def extract_date_from_filename(filename):
    """Extrair ano/mês de padrões como YYYY_MM.xlsx"""
    stem = filename.stem
    match = re.match(r'(\d{2,4})_(\d{2})', stem)
    if match:
        ano, mes = match.groups()
        if len(ano) == 2:
            ano = '20' + ano
        return f"{ano}-{mes}", (int(ano), int(mes))
    return None, None

# Varrer apenas arquivos XLSX (Prestações de Contas)
xlsx_files = list(XLSX_DIR.glob("*.xlsx")) if XLSX_DIR.exists() else []

# Organizar em tabela de controle
controle = []

for f in sorted(xlsx_files):
    mes_ano, data_tuple = extract_date_from_filename(f)
    if mes_ano:
        controle.append({
            'arquivo': f.name,
            'caminho': str(f),
            'mes_ano': mes_ano,
            'tipo': 'XLSX (prestação)',
            'tamanho_mb': f.stat().st_size / (1024*1024)
        })

df_controle = pd.DataFrame(controle).sort_values('mes_ano').reset_index(drop=True)

print(f"\n📋 COBERTURA DE PRESTAÇÕES DE CONTAS:\n")
print(df_controle.to_string(index=False))
print(f"\nTotal: {len(df_controle)} arquivos")
print(f"Período: {df_controle['mes_ano'].min()} até {df_controle['mes_ano'].max()}")


📋 COBERTURA DE PRESTAÇÕES DE CONTAS:

     arquivo                                                              caminho mes_ano             tipo  tamanho_mb
2022_05.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_05.xlsx 2022-05 XLSX (prestação)    0.022434
2022_06.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_06.xlsx 2022-06 XLSX (prestação)    0.023280
2022_07.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_07.xlsx 2022-07 XLSX (prestação)    0.024342
2022_08.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_08.xlsx 2022-08 XLSX (prestação)    0.024735
2022_09.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_09.xlsx 2022-09 XLSX (prestação)    0.024455
2022_10.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_10.xlsx 2022-10 XLSX (prestação)    0.024429
2022_11.xlsx /Users/chris/Documents/analisis/exports/hojas/prestacao/2022_11.xlsx 2022-11 XLSX (prestação)    0.024192
2022_12.x

In [3]:
## 3. Carregar e consolidar dados XLSX (Prestações de Contas 2022–2026)

def _is_skip_row(evento):
    """
    Determinar se uma linha deve ser removida (subtotal, seção ou header).
    
    Linhas a remover:
    - Vazias ou com NaN
    - Contêm '**' (seções: ** Pessoal ** etc)
    - Contêm 'SUBTOTAL' (subtotal final)
    - Começam com 'OUTROS EVENTOS' (subtotal de grupo de primeira seção)
    """
    if pd.isna(evento):
        return True
    
    e = str(evento).strip().upper()
    
    if e == '':
        return True
    if '**' in e:  # Seções como ** Pessoal **
        return True
    if 'SUBTOTAL' in e:  # Linha de total
        return True
    if e.startswith('OUTROS EVENTOS'):  # Subtotal de primeiro grupo (Receitas/Despesas)
        return True
    
    return False


def load_xlsx_prestacao(filepath):
    """Carregar prestação XLSX — abas Receitas e Despesas com Evento/Valor"""
    try:
        resultado_lista = []
        
        for sheet in ['Receitas', 'Despesas']:
            # Ler com header=1 (pula linha de título)
            df = pd.read_excel(filepath, sheet_name=sheet, header=1)
            
            # Padronizar colunas (primeira coluna = Evento, segunda = Valor)
            df.columns = ['evento', 'valor']
            
            # Filtrar: remover linhas vazias, subtotais e seções
            df = df[~df['evento'].apply(_is_skip_row)].copy()
            df['evento'] = df['evento'].astype(str).str.strip()
            
            # Adicionar tipo
            df['tipo'] = 'RECEITA' if sheet == 'Receitas' else 'DESPESA'
            
            resultado_lista.append(df[['evento', 'valor', 'tipo']])
        
        if resultado_lista:
            return pd.concat(resultado_lista, ignore_index=True)
        else:
            return None
            
    except Exception as e:
        print(f"  ⚠️  Erro ao carregar {filepath.name}: {e}")
        return None

# Carregar todos os arquivos XLSX
print("\\n🔄 CARREGANDO PRESTAÇÕES DE CONTAS (XLSX)...\\n")

dados_xlsx = []

for f in sorted(XLSX_DIR.glob("*.xlsx")) if XLSX_DIR.exists() else []:
    mes_ano, _ = extract_date_from_filename(f)
    df = load_xlsx_prestacao(f)
    if df is not None and len(df) > 0:
        df['mes_ano'] = mes_ano
        dados_xlsx.append(df)
        print(f"  ✓ {f.name}: {len(df)} registros")

# Consolidar
print(f"\\n🔗 Consolidando...")

if dados_xlsx:
    df_base = pd.concat(dados_xlsx, ignore_index=True, sort=False)
    print(f"✅ Base consolidada: {len(df_base)} registros | {df_base['mes_ano'].nunique()} períodos")
    print(f"   Período: {df_base['mes_ano'].min()} até {df_base['mes_ano'].max()}")
else:
    print("❌ Nenhum arquivo foi carregado com sucesso")
    df_base = pd.DataFrame()

\n🔄 CARREGANDO PRESTAÇÕES DE CONTAS (XLSX)...\n
  ✓ 2022_05.xlsx: 35 registros
  ✓ 2022_06.xlsx: 42 registros
  ✓ 2022_07.xlsx: 49 registros
  ✓ 2022_08.xlsx: 49 registros
  ✓ 2022_09.xlsx: 48 registros
  ✓ 2022_10.xlsx: 45 registros
  ✓ 2022_11.xlsx: 43 registros
  ✓ 2022_12.xlsx: 51 registros
  ✓ 2023_01.xlsx: 47 registros
  ✓ 2023_02.xlsx: 54 registros
  ✓ 2023_03.xlsx: 51 registros
  ✓ 2023_04.xlsx: 50 registros
  ✓ 2023_05.xlsx: 46 registros
  ✓ 2023_07.xlsx: 49 registros
  ✓ 2023_08.xlsx: 52 registros
  ✓ 2023_09.xlsx: 51 registros
  ✓ 2023_10.xlsx: 48 registros
  ✓ 2023_11.xlsx: 46 registros
  ✓ 2023_12.xlsx: 40 registros
  ✓ 2024_01.xlsx: 40 registros
  ✓ 2024_02.xlsx: 40 registros
  ✓ 2024_03.xlsx: 44 registros
  ✓ 2024_04.xlsx: 43 registros
  ✓ 2024_05.xlsx: 39 registros
  ✓ 2024_06.xlsx: 35 registros
  ✓ 2024_07.xlsx: 43 registros
  ✓ 2024_08.xlsx: 41 registros
  ✓ 2024_09.xlsx: 40 registros
  ✓ 2024_10.xlsx: 43 registros
  ✓ 2024_11.xlsx: 38 registros
  ✓ 2024_12.xlsx: 45 r

In [4]:
## 4. Limpar e normalizar valores

def clean_currency(val):
    """Converter string de moeda BR para float"""
    if pd.isna(val) or val == '' or val == '-':
        return None
    if isinstance(val, (int, float)):
        return float(val)
    
    val_str = str(val).strip()
    val_str = val_str.replace('R$', '').strip()
    # Inverter separadores (BR: . é mil, , é decimal)
    val_str = val_str.replace('.', 'TEMP')
    val_str = val_str.replace(',', '.')
    val_str = val_str.replace('TEMP', '')
    
    try:
        return float(val_str)
    except:
        return None

# Limpar valores
df_base['valor'] = df_base['valor'].apply(clean_currency)

# Normalizar textos
for col in ['evento', 'tipo']:
    if col in df_base.columns:
        df_base[col] = df_base[col].str.strip().str.upper()

print("✅ Limpeza concluída:")
print(f"  Valores normalizados (BR format)")
print(f"  Textos em uppercase")

# Amostra
print("\n📊 Amostra dos dados:\n")
print(df_base.head(10).to_string())

✅ Limpeza concluída:
  Valores normalizados (BR format)
  Textos em uppercase

📊 Amostra dos dados:

                         evento     valor     tipo  mes_ano
0                REC.CONDOMÍNIO  41248.84  RECEITA  2022-05
1                 FUNDO RESERVA   1927.26  RECEITA  2022-05
2              REC.TAXA USO BOX    190.00  RECEITA  2022-05
3                   FUNDO OBRAS     22.98  RECEITA  2022-05
4                    REC. MULTA    108.19  RECEITA  2022-05
5           REC.MULTA+C.M.+JRS.     23.31  RECEITA  2022-05
6                    PG.SALÁRIO   4637.14  DESPESA  2022-05
7                       PG.FGTS    268.34  DESPESA  2022-05
8               PG.ADTO.SALÁRIO    600.00  DESPESA  2022-05
9  PG.VALE REFEIÇÃO/ALIMENTAÇÃO    411.00  DESPESA  2022-05


In [5]:
## 5. Validar e resumir dados

print("\n🔍 RESUMO DA BASE\n")

# Distribuição por tipo
print(f"Distribuição por tipo:")
print(df_base['tipo'].value_counts())

# Total por mês (últimos 12)
print(f"\n\nTotal por mês (últimos 12):")
total_mes = df_base[df_base['valor'].notna()].groupby('mes_ano')['valor'].agg(['sum', 'count']).round(2)
for mes, row in total_mes.tail(12).iterrows():
    print(f"  {mes}: R$ {row['sum']:>12,.2f} ({int(row['count']):>3} registros)")

# Valor total
print(f"\n\nResumo geral:")
val_total = df_base['valor'].dropna().sum()
print(f"  Valor total: R$ {val_total:,.2f}")
print(f"  Registros com valor: {len(df_base[df_base['valor'].notna()])}")
print(f"  Período: {df_base['mes_ano'].min()} → {df_base['mes_ano'].max()}")


🔍 RESUMO DA BASE

Distribuição por tipo:
tipo
DESPESA    1776
RECEITA     455
Name: count, dtype: int64


Total por mês (últimos 12):
  2025-07: R$   182,733.45 ( 55 registros)
  2025-08: R$   147,417.02 ( 53 registros)
  2025-09: R$   136,234.09 ( 47 registros)
  2025-10: R$   139,139.04 ( 45 registros)
  2025-11: R$   145,384.16 ( 44 registros)
  2025-12: R$   141,547.49 ( 44 registros)
  2026-01: R$   154,788.84 ( 52 registros)
  2026-02: R$   140,494.19 ( 42 registros)
  2026-03: R$   144,921.32 ( 45 registros)
  2026-04: R$   142,486.52 ( 43 registros)
  2026-05: R$   147,485.69 ( 50 registros)
  2026-06: R$   146,918.49 ( 52 registros)


Resumo geral:
  Valor total: R$ 6,655,500.31
  Registros com valor: 2231
  Período: 2022-05 → 2026-06


In [6]:
## 6. Top categorias por valor

print("\n📊 TOP 20 CATEGORIAS POR VALOR TOTAL\n")

top_categorias = df_base[df_base['valor'].notna()].groupby('evento')['valor'].agg(['sum', 'count']).round(2)
top_categorias = top_categorias.sort_values('sum', ascending=False).head(20)

for evento, row in top_categorias.iterrows():
    print(f"  {str(evento)[:50]:<50} R$ {row['sum']:>12,.2f} ({int(row['count']):>2}x)")


📊 TOP 20 CATEGORIAS POR VALOR TOTAL

  REC.CONDOMÍNIO                                     R$ 3,030,768.40 (53x)
  PG.SERV.PORTARIA                                   R$   895,385.69 (49x)
  ÁGUA/ESGOTO                                        R$   303,912.50 (33x)
  PG.SERV.LIMP.                                      R$   248,334.70 (48x)
  PG.SÍNDICO PROF.                                   R$   151,888.40 (45x)
  ÁGUA                                               R$   133,075.34 (14x)
  PG.SALÁRIO                                         R$   132,916.32 (49x)
  PG.DARF INSS(E-SOCIAL/REINF)                       R$   126,657.42 (47x)
  PG.REFORMA                                         R$   119,681.57 (13x)
  PG.CEEE                                            R$    91,076.08 (45x)
  FUNDO OBRAS                                        R$    86,089.56 (49x)
  PG.IMPERMEABILIZAÇÃO                               R$    73,467.99 ( 6x)
  VALOR                                              R$    63,

In [7]:
## 7. Exportar dados consolidados

# Exportar base consolidada (compatível com analise_prestacao_de_contas.ipynb)
output_file = CSV_DIR / 'prestacoes.csv'
df_base.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✅ Base consolidada exportada:")
print(f"  Arquivo: {output_file.name}")
print(f"  Tamanho: {output_file.stat().st_size / (1024*1024):.2f} MB")
print(f"  Registros: {len(df_base)}")
print(f"  Períodos: {df_base['mes_ano'].nunique()} ({df_base['mes_ano'].min()} → {df_base['mes_ano'].max()})")
print(f"  Valor total: R$ {df_base['valor'].dropna().sum():,.2f}")

# Exportar tabela de controle
controle_file = CSV_DIR / 'cobertura_prestacoes.csv'
df_controle.to_csv(controle_file, index=False, encoding='utf-8')
print(f"\n📋 Tabela de controle: {controle_file.name}")


✅ Base consolidada exportada:
  Arquivo: prestacoes.csv
  Tamanho: 0.09 MB
  Registros: 2231
  Períodos: 49 (2022-05 → 2026-06)
  Valor total: R$ 6,655,500.31

📋 Tabela de controle: cobertura_prestacoes.csv


In [21]:
## 8. Próximas etapas

print("\n" + "="*60)
print("✅ PIPELINE DE INGESTÃO CONCLUÍDO")
print("="*60)

print("\n📋 Saída gerada:")
print(f"  • prestacoes.csv — base consolidada (2022-2026)")
print(f"  • cobertura_prestacoes.csv — tabela de controle")

print("\n🎯 Próximo passo:")
print(f"  Execute: analise_prestacao_de_contas.ipynb")
print(f"  (irá carregar prestacoes.csv e fazer análises completas)")

print("\n🔄 Como adicionar novos meses:")
print(f"  1. Baixe XLSX do sistema ClienteOnline")
print(f"  2. Coloque em: exports/hojas/prestacao/")
print(f"  3. Nomeie como: YYYY_MM.xlsx (ex: 2026_07.xlsx)")
print(f"  4. Execute este notebook novamente")
print(f"  5. Base será atualizada automaticamente")
print()



✅ PIPELINE DE INGESTÃO CONCLUÍDO

📋 Saída gerada:
  • prestacoes.csv — base consolidada (2022-2026)
  • cobertura_prestacoes.csv — tabela de controle

🎯 Próximo passo:
  Execute: analise_prestacao_de_contas.ipynb
  (irá carregar prestacoes.csv e fazer análises completas)

🔄 Como adicionar novos meses:
  1. Baixe XLSX do sistema ClienteOnline
  2. Coloque em: exports/hojas/prestacao/
  3. Nomeie como: YYYY_MM.xlsx (ex: 2026_07.xlsx)
  4. Execute este notebook novamente
  5. Base será atualizada automaticamente



## ✅ Conclusão

Este notebook processa **apenas XLSX** (Prestações de Contas 2022–2026) e exporta um CSV consolidado para análise.

### Arquitetura simplificada:

```
consolidacao_xls.ipynb  (ingestão + limpeza)
        ↓ prestacoes.csv
analise_prestacao_de_contas.ipynb  (análises + insights)
```

### Como usar:
1. **Executar este notebook** quando houver novos XLSX
2. **Executar analise_prestacao_de_contas.ipynb** para análises completas


# 📊 Consolidação de Prestações de Contas — XLSX (2022–2026)

**Objetivo**: Pipeline simples e escalável para carregar, validar e normalizar **todos os XLSX de prestações de contas** do condomínio.

**Input**: Arquivos XLSX em `exports/hojas/prestacao/` (2022-2026)  
**Output**: `prestacoes.csv` — base consolidada e limpa

**Próximo**: Alimenta `analise_prestacao_de_contas.ipynb` para análises aprofundadas.
